In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/Mate

cleaned = pd.read_csv("ab_test.csv")

df = pd.DataFrame(cleaned)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Mate


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Lithuania,mobile,Europe,Organic Search,2,2,new account,1
1,2020-11-01,El Salvador,desktop,Americas,Social Search,2,1,new account,1
2,2020-11-01,Slovakia,mobile,Europe,Paid Search,2,2,new account,1
3,2020-11-01,Lithuania,desktop,Europe,Paid Search,2,2,new account,1
4,2020-11-02,North Macedonia,desktop,Europe,Direct,2,1,new account,1


In [ ]:
metrics = [
    'add_payment_info',
    'add_shipping_info',
    'begin_checkout',
    'new account'
]

denominator_event = 'session'


dimensions = [
    'total',
    'continent',
    'country',
    'device',
    'channel'
]


results = []

for d in dimensions:

    if d == 'total':
        index_cols = ['test', 'event_name']
    else:
        index_cols = ['test', d, 'event_name']

    pivot = (
        df[df['event_name'].isin(metrics + [denominator_event])]
        .pivot_table(
            index=index_cols,
            columns='test_group',
            values='value',
            aggfunc='sum',
            fill_value=0
        )
        .reset_index()
    )

    sessions = (
        pivot[pivot['event_name'] == denominator_event]
        .rename(columns={
            1: 'denominator_control',
            2: 'denominator_test'
        })
        .drop(columns='event_name')
    )

    numerators = pivot[pivot['event_name'].isin(metrics)]

    merge_keys = index_cols[:-1]

    final = (
        numerators
        .merge(
            sessions,
            on=merge_keys,
            how='left'
        )
       .rename(columns={
            1: 'numerator_control',
            2: 'numerator_test'
       })
   )

    final['conversion_rate_control'] = (
        final['numerator_control'] / final['denominator_control']
    )
    final['conversion_rate_test'] = (
        final['numerator_test'] / final['denominator_test']
    )
    final['denominator_event'] = denominator_event

    final['metric_change'] = np.where(
    final['conversion_rate_control'] == 0,
    np.nan,
    ((final['conversion_rate_test']) / final['conversion_rate_control'] -1) * 100
)

    z_stats = []
    p_values = []

    for _, row in final.iterrows():
        successes = [row['numerator_test'], row['numerator_control']]
        trials = [row['denominator_test'], row['denominator_control']]

        z, p = proportions_ztest(successes, trials)
        z_stats.append(z)
        p_values.append(p)

    final['z_stat'] = z_stats
    final['p_value'] = p_values
    final['significant'] = final['p_value'] < 0.05

    final['dimension'] = d

    if d == 'total':
        final['dimension_value'] = 'all'
    else:
        final['dimension_value'] = final[d].astype(str)
        final = final.drop(columns=[d])

    final = final.rename(columns={
        'test': 'test_number',
        'event_name': 'metric'
    })

    results.append(final)


final_result = pd.concat(results, ignore_index=True)


final_result = final_result[[
    'test_number',
    'metric',
    'denominator_event',
    'dimension',
    'dimension_value',
    'numerator_control',
    'denominator_control',
    'conversion_rate_control',
    'numerator_test',
    'denominator_test',
    'conversion_rate_test',
    'metric_change',
    'z_stat',
    'p_value',
    'significant'
]]

final_result


/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)


test_group,test_number,metric,denominator_event,dimension,dimension_value,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant
0,1,add_payment_info,session,total,all,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping_info,session,total,all,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout,session,total,all,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
3,1,new account,session,total,all,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info,session,total,all,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1806,4,new account,session,channel,Social Search,667,7961,0.083783,653,8056,0.081058,-3.253444,-0.627241,0.530501,False
1807,4,add_payment_info,session,channel,Undefined,496,5716,0.086774,566,5862,0.096554,11.270787,1.822812,0.068332,False
1808,4,add_shipping_info,session,channel,Undefined,640,5716,0.111966,666,5862,0.113613,1.470701,0.280026,0.779457,False
1809,4,begin_checkout,session,channel,Undefined,1578,5716,0.276067,1600,5862,0.272944,-1.131171,-0.376454,0.706579,False


In [ ]:
final_result.to_csv('ab_test_results.csv', index=False)

In [ ]:
final_result.head(10)

test_group,test_number,metric,denominator_event,dimension,dimension_value,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant
0,1,add_payment_info,session,total,all,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping_info,session,total,all,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout,session,total,all,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
3,1,new account,session,total,all,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info,session,total,all,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
5,2,add_shipping_info,session,total,all,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,False
6,2,begin_checkout,session,total,all,4262,50637,0.084168,4313,50244,0.085841,1.988164,0.952898,0.340642,False
7,2,new account,session,total,all,4165,50637,0.082252,4184,50244,0.083274,1.241934,0.588793,0.556000,False
8,3,add_payment_info,session,total,all,3623,70047,0.051722,3697,70439,0.052485,1.474630,0.643172,0.520112,False
9,3,add_shipping_info,session,total,all,5298,70047,0.075635,5188,70439,0.073652,-2.621211,-1.413727,0.157442,False


#[Tableau](https://public.tableau.com/app/profile/daryna.nakonechna/viz/ABTestingAnalyticsStatisticalValidation/ViewSignificance#3)